**Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

**Exploración y Curación de Datos**

*Edición 2026*

----

# Trabajo práctico entregable - parte 2

En esta notebook, vamos a cargar el conjunto de datos de [la compentencia Kaggle](https://www.kaggle.com/dansbecker/melbourne-housing-snapshot) sobre estimación de precios de ventas de propiedades en Melbourne, Australia.

Utilizaremos el conjunto de datos reducido producido por [DanB](https://www.kaggle.com/dansbecker). Hemos subido una copia a un servidor de la Universidad Nacional de Córdoba para facilitar su acceso remoto.

In [ ]:
import matplotlib.pyplot as plt
import numpy
import pandas

import seaborn
seaborn.set_context('talk')

from sqlalchemy import create_engine, text

In [ ]:
import plotly
plotly.__version__


'5.24.1'

In [ ]:
melb_df = pandas.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv')
melb_df[:3]

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


## Ejercicio 1 SQL:

1. Crear una base de datos en SQLite utilizando la libreria [SQLalchemy](https://stackoverflow.com/questions/2268050/execute-sql-from-file-in-sqlalchemy).
https://docs.sqlalchemy.org/en/14/core/engines.html#sqlite

2. Ingestar los datos provistos en 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv' en una tabla y el dataset generado en clase con datos de airbnb y sus precios por codigo postal en otra.

3. Validar tipos de columnas antes de guardar. Usá `df.dtypes` para ver los tipos actuales. Prestá especial atención a columnas como `Date` y `Price`: por ejemplo, `Date` puede estar como string en vez de datetime, y `Price` puede venir como string o float. El método `to_sql()` infiere tipos automáticamente, pero puede fallar si los tipos no son los esperados.

4. Implementar consultas en SQL que respondan con la siguiente información:

    - cantidad de registros totales por `Regionname`.
    - cantidad de registros totales por `Suburb` y `Regionname`.
    - Consulta con filtro: ¿Cuántas propiedades hay por `Regionname` con más de 2 habitaciones?
    - Agregación condicional: ¿Cuál es el precio promedio de propiedades según tipo (`Type`) y `Regionname`?
    - Orden y límites: Mostrá el top 5 barrios con propiedades más caras en promedio.

5. Combinar los datasets de ambas tablas ingestadas utilizando el comando JOIN de SQL para obtener un resultado similar a lo realizado con Pandas en clase.

6. Agregar una celda de validación posterior al JOIN con assertions o validación de esquema. Como mínimo, verificá que el número de filas no cambió, que no aparecieron nulos inesperados y que los rangos de variables agregadas sean razonables. Esta validación implementa dimensiones básicas de calidad de datos como validez, completitud e integridad.



In [ ]:
# Crear la base de datos SQLite
engine = create_engine('sqlite:///melbourne_airbnb.sqlite')

In [ ]:
# Seleccionamos las columnas de interes e importamos la base Airbnb
interesting_cols = [
    'description', 'neighborhood_overview',
    'street', 'neighborhood', 'city', 'suburb', 'state', 'zipcode',
    'price', 'weekly_price', 'monthly_price',
    'latitude', 'longitude',
]

airbnb_df = pandas.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/cleansed_listings_dec18.csv',
    usecols=interesting_cols
)

airbnb_df[:3]

/tmp/ipykernel_2251/106389825.py:9: DtypeWarning: Columns (35) have mixed types. Specify dtype option on import or set low_memory=False.
  airbnb_df = pandas.read_csv(


,description,neighborhood_overview,street,neighborhood,city,suburb,state,zipcode,latitude,longitude,price,weekly_price,monthly_price
0,"House: Clean, New, Modern, Quite, Safe. 10Km f...",Very safe! Family oriented. Older age group.,"Bulleen, VIC, Australia",Balwyn North,Manningham,Bulleen,VIC,3105,-37.772684,145.092133,60,NaN,NaN
1,A large air conditioned room with queen spring...,This hip area is a crossroads between two grea...,"Brunswick East, VIC, Australia",Brunswick,Moreland,Brunswick East,VIC,3057,-37.766505,144.980736,35,200.0,803.0
2,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,A stay at our apartment means you can enjoy so...,"St Kilda, VIC, Australia",St Kilda,Port Phillip,St Kilda,VIC,3182,-37.859755,144.977369,159,1253.0,4452.0


Primero hay que validar el tipo de datos

In [ ]:
melb_df.dtypes

,0
Suburb,object
Address,object
Rooms,int64
Type,object
Price,float64
Method,object
SellerG,object
Date,object
Distance,float64
Postcode,float64


In [ ]:
melb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

In [ ]:
airbnb_df.dtypes

,0
description,object
neighborhood_overview,object
street,object
neighborhood,object
city,object
suburb,object
state,object
zipcode,object
latitude,float64
longitude,float64


In [ ]:
airbnb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22895 entries, 0 to 22894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   description            22563 non-null  object 
 1   neighborhood_overview  14424 non-null  object 
 2   street                 22895 non-null  object 
 3   neighborhood           17082 non-null  object 
 4   city                   22895 non-null  object 
 5   suburb                 22872 non-null  object 
 6   state                  22834 non-null  object 
 7   zipcode                22753 non-null  object 
 8   latitude               22895 non-null  float64
 9   longitude              22895 non-null  float64
 10  price                  22895 non-null  int64  
 11  weekly_price           2524 non-null   float64
 12  monthly_price          1891 non-null   float64
dtypes: float64(4), int64(1), object(8)
memory usage: 2.3+ MB


In [ ]:
# Date debe ser fecha, no texto
melb_df['Date'] = pandas.to_datetime(
    melb_df['Date'],
    dayfirst=True,
    errors='coerce'
)

# Postcode es un código postal, por lo tanto se trata como categórica
melb_df['Postcode'] = melb_df['Postcode'].astype('Int64').astype('str')

# Propertycount, Bathroom y Car son conteos, por lo tanto se convierten a enteros
melb_df['Propertycount'] = melb_df['Propertycount'].astype('Int64')
melb_df['Bathroom'] = melb_df['Bathroom'].astype('Int64')
melb_df['Car'] = melb_df['Car'].astype('Int64')

In [ ]:
melb_df[['Date', 'Postcode', 'Propertycount', 'Bathroom', 'Car', 'Price']].dtypes

,0
Date,datetime64[ns]
Postcode,object
Propertycount,Int64
Bathroom,Int64
Car,Int64
Price,float64


In [ ]:
# zipcode es un código postal, por lo tanto se trata como categórica
airbnb_df['zipcode'] = pandas.to_numeric(
    airbnb_df['zipcode'],
    errors='coerce'
).astype('Int64').astype('string')

In [ ]:
# Guardar los DataFrames como tablas en SQLite

melb_df.to_sql(
    'melbourne',
    con=engine,
    if_exists='replace',
    index=False
)

airbnb_df.to_sql(
    'airbnb',
    con=engine,
    if_exists='replace',
    index=False
)

22895

In [ ]:
# Cantidad de registros totales por Regionname
query1 = """
SELECT Regionname, COUNT(1) AS cantidad_registros
FROM melbourne
GROUP BY Regionname
"""

# Cantidad de registros totales por Suburb y Regionname
query2 = """
SELECT Suburb, Regionname, COUNT(1) AS cantidad_registros
FROM melbourne
GROUP BY Suburb, Regionname
"""

# Consulta con filtro: ¿Cuántas propiedades hay por Regionname con más de 2 habitaciones?
query3 = """
SELECT Regionname, COUNT(1) AS cantidad_propiedades
FROM melbourne
WHERE Rooms > 2
GROUP BY Regionname
"""

# Agregación condicional: ¿Cuál es el precio promedio de propiedades según tipo (Type) y Regionname?
query4 = """
SELECT Type, Regionname, AVG(Price) AS precio_promedio
FROM melbourne
GROUP BY Type, Regionname
"""

# Orden y límites: Mostrá el top 5 barrios con propiedades más caras en promedio.
query5 = """
SELECT Suburb, AVG(Price) AS precio_promedio
FROM melbourne
GROUP BY Suburb
ORDER BY AVG(Price) DESC
LIMIT 5
"""

queries = [query1, query2, query3, query4, query5]

In [ ]:
with engine.connect() as con:
    for query in queries:
        rs = con.execute(text(query))
        print(query)
        for row in rs:
            print(row)

        print('\n\n')


SELECT Regionname, COUNT(1) AS cantidad_registros
FROM melbourne
GROUP BY Regionname

('Eastern Metropolitan', 1471)
('Eastern Victoria', 53)
('Northern Metropolitan', 3890)
('Northern Victoria', 41)
('South-Eastern Metropolitan', 450)
('Southern Metropolitan', 4695)
('Western Metropolitan', 2948)
('Western Victoria', 32)




SELECT Suburb, Regionname, COUNT(1) AS cantidad_registros
FROM melbourne
GROUP BY Suburb, Regionname

('Abbotsford', 'Northern Metropolitan', 56)
('Aberfeldie', 'Western Metropolitan', 44)
('Airport West', 'Western Metropolitan', 67)
('Albanvale', 'Western Metropolitan', 6)
('Albert Park', 'Southern Metropolitan', 69)
('Albion', 'Western Metropolitan', 41)
('Alphington', 'Northern Metropolitan', 34)
('Altona', 'Western Metropolitan', 74)
('Altona Meadows', 'Western Metropolitan', 6)
('Altona North', 'Western Metropolitan', 56)
('Ardeer', 'Western Metropolitan', 3)
('Armadale', 'Southern Metropolitan', 95)
('Ascot Vale', 'Western Metropolitan', 130)
('Ashburton', 

In [ ]:
query_join = """
SELECT
    m.*,
    a.zipcode AS airbnb_zipcode,
    a.airbnb_price_mean,
    a.airbnb_record_count,
    a.airbnb_weekly_price_mean,
    a.airbnb_monthly_price_mean
FROM melbourne AS m
LEFT JOIN (
    SELECT
        zipcode,
        AVG(price) AS airbnb_price_mean,
        COUNT(1) AS airbnb_record_count,
        AVG(weekly_price) AS airbnb_weekly_price_mean,
        AVG(monthly_price) AS airbnb_monthly_price_mean
    FROM airbnb
    GROUP BY zipcode
) AS a
ON m.Postcode = a.zipcode
"""

merged_sales_df = pandas.read_sql(query_join, con=engine)

merged_sales_df.head()

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount,airbnb_zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,2016-12-03 00:00:00.000000,2.5,3067,...,Yarra,-37.7996,144.9984,Northern Metropolitan,4019,3067,130.624031,258.0,605.152174,2187.032258
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,2016-02-04 00:00:00.000000,2.5,3067,...,Yarra,-37.8079,144.9934,Northern Metropolitan,4019,3067,130.624031,258.0,605.152174,2187.032258
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,2017-03-04 00:00:00.000000,2.5,3067,...,Yarra,-37.8093,144.9944,Northern Metropolitan,4019,3067,130.624031,258.0,605.152174,2187.032258
3,Abbotsford,40 Federation La,3,h,850000.0,PI,Biggin,2017-03-04 00:00:00.000000,2.5,3067,...,Yarra,-37.7969,144.9969,Northern Metropolitan,4019,3067,130.624031,258.0,605.152174,2187.032258
4,Abbotsford,55a Park St,4,h,1600000.0,VB,Nelson,2016-06-04 00:00:00.000000,2.5,3067,...,Yarra,-37.8072,144.9941,Northern Metropolitan,4019,3067,130.624031,258.0,605.152174,2187.032258


In [ ]:
print(f"Filas antes: {len(melb_df)}")
print(f"Filas después: {len(merged_sales_df)}")

print("Nulos en Price:", merged_sales_df["Price"].isna().sum())
print("Nulos en Regionname:", merged_sales_df["Regionname"].isna().sum())
print("Nulos en airbnb_price_mean:", merged_sales_df["airbnb_price_mean"].isna().sum())
print("Nulos en airbnb_record_count:", merged_sales_df["airbnb_record_count"].isna().sum())
print("Nulos en airbnb_weekly_price_mean:", merged_sales_df["airbnb_weekly_price_mean"].isna().sum())
print("Nulos en airbnb_monthly_price_mean:", merged_sales_df["airbnb_monthly_price_mean"].isna().sum())

Filas antes: 13580
Filas después: 13580
Nulos en Price: 0
Nulos en Regionname: 0
Nulos en airbnb_price_mean: 20
Nulos en airbnb_record_count: 20
Nulos en airbnb_weekly_price_mean: 875
Nulos en airbnb_monthly_price_mean: 1268


In [ ]:
assert len(merged_sales_df) == len(melb_df), "El JOIN cambió el número de filas"

assert merged_sales_df["Price"].isna().sum() == 0, "Hay nulos inesperados en Price"

assert merged_sales_df["Regionname"].isna().sum() == 0, "Hay nulos inesperados en Regionname"

assert merged_sales_df["airbnb_price_mean"].dropna().between(0, 10000).all(), \
    "Hay valores fuera de rango en airbnb_price_mean"

assert merged_sales_df["airbnb_record_count"].dropna().ge(1).all(), \
    "Hay valores inválidos en airbnb_record_count"

assert merged_sales_df["airbnb_weekly_price_mean"].dropna().between(0, 50000).all(), \
    "Hay valores fuera de rango en airbnb_weekly_price_mean"

assert merged_sales_df["airbnb_monthly_price_mean"].dropna().between(0, 200000).all(), \
    "Hay valores fuera de rango en airbnb_monthly_price_mean"

## Ejercicio 2 - Pandas:

Este ejercicio usa el archivo `airbnb_price_by_zipcode.csv` generado en el notebook `02.1 Combinación de datasets.ipynb`. Si no lo tenés, generarlo primero antes de comenzar esta parte.

1. Seleccionar un subconjunto de columnas que les parezcan relevantes al problema de predicción del valor de la propiedad. Justificar explicitamente las columnas seleccionadas y las que no lo fueron.
  1. Valores faltantes: ¿Qué porcentaje de filas tienen al menos un valor faltante?
  2. Mostrar la dispersión o distribución de las columnas seleccionadas.
 3. Eliminar los valores extremos que no sean relevantes para la predicción de valores de las propiedades.
 4. Mostrar visualmente los valores extremos que eliminás


2. Agregar información adicional respectiva al entorno de una propiedad a partir del [conjunto de datos de AirBnB](https://www.kaggle.com/tylerx/melbourne-airbnb-open-data?select=cleansed_listings_dec18.csv) utilizado en el práctico.
  1. Seleccionar qué variables agregar y qué combinaciones aplicar a cada una. Por ejemplo, pueden utilizar solo la columna `price`, o aplicar múltiples transformaciones como la mediana (¿por qué no la media?) o el mínimo.
  2. Utilizar la variable zipcode para unir los conjuntos de datos. Sólo incluir los zipcodes que tengan una cantidad mínima de registros (a elección) como para que la información agregada sea relevante.
  3. Mostrar un gráfico zipcode vs airbnb_price_median.
  4. Investigar al menos otras 2 variables que puedan servir para combinar los datos, y justificar si serían adecuadas o no. Pueden asumir que cuentan con la ayuda de anotadores expertos para encontrar equivalencias entre barrios o direcciones, o que cuentan con algoritmos para encontrar las n ubicaciones más cercanas a una propiedad a partir de sus coordenadas geográficas. **NO** es necesario que realicen la implementación. Si tuvieras que entrevistar a un experto inmobiliario para mapear barrios entre datasets, ¿qué 3 preguntas le harías para validar esa correspondencia?
  5. Si las coordenadas geoespaciales estuvieran disponibles, como las usarian?

Pueden leer otras columnas del conjunto de AirBnB además de las que están en `interesting_cols`, si les parecen relevantes.

¿Qué cosas no están en los datos que te gustaría tener para predecir mejor el precio de una propiedad?


### Criterios de evaluación
Se evaluará principalmente:
- claridad del código,
- justificación de las decisiones de curación,
- coherencia entre el análisis realizado y las conclusiones,
- presencia de validaciones después de operaciones críticas como merges o cargas a base.

No se espera una única solución correcta, pero sí que las decisiones estén justificadas y sean consistentes con los datos.


## Ejercicio 3:

Crear y guardar un nuevo conjunto de datos con todas las transformaciones realizadas anteriormente.

## Ejercicios opcionales:

El notebook `02.2 ETLs-DAGs.ipynb` tiene un esqueleto de referencia para guiarse.

1. Armar un script en python (archivo .py) [ETL](https://towardsdatascience.com/what-to-log-from-python-etl-pipelines-9e0cfe29950e) que corra los pasos de extraccion, transformacion y carga, armando una funcion para cada etapa del proceso y luego un main que corra todos los pasos requeridos.

2. Armar un DAG en Apache Airflow que corra el ETL. (https://airflow.apache.org/docs/apache-airflow/stable/tutorial.html)

3. Bonus: embeddings y búsqueda semántica con descripciones de AirBnB.
   - Usar `sentence-transformers` para codificar descripciones textuales de propiedades.
   - Tomar un subconjunto chico de descripciones, calcular embeddings y encontrar el par más similar con similitud coseno.
   - Reflexionar: ¿por qué este resultado no se puede lograr con `LIKE '%keyword%'` en SQL? ¿Qué pasa si dos propiedades son similares pero usan palabras distintas? ¿Qué representan los 384 números del embedding?

4. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?


5. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?

Ejemplo conceptual:


In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# Valores únicos con posibles inconsistencias
council_values = melb_df['CouncilArea'].dropna().unique().tolist()

message = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": f"""Estos son los valores únicos de la columna CouncilArea en un dataset de propiedades de Melbourne:
{council_values}

Identificá: (1) duplicados con distinta capitalización o spelling,
 (2) valores que parecen errores, (3) valores que podrían agruparse.
Respondé en JSON con la estructura: {{"estandarizado": {{"valor_original": "valor_correcto"}}}}"""
    }]
)

mapping = json.loads(message.content[0].text)
melb_df['CouncilArea_clean'] = melb_df['CouncilArea'].map(
    mapping.get('estandarizado', {})
).fillna(melb_df['CouncilArea'])


ModuleNotFoundError: No module named 'anthropic'